# Hohmann and bi-elliptic orbit transfers

*astro · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/orbit_transfer)

How much $\Delta v$ does it take to move a spacecraft from one circular orbit to another? The vis-viva equation $v^2 = \mu(2/r - 1/a)$ prices the two-burn Hohmann transfer and the three-burn bi-elliptic transfer, and one batched RK4 integration with impulsive burns at the apsides checks that each spacecraft really ends up on the target orbit.

**Vector form**

$$\ddot{\mathbf r} = -\frac{\mu}{r^3}\,\mathbf r,\qquad v^2 = \mu\Big(\frac{2}{r} - \frac{1}{a}\Big),\qquad \Delta\mathbf v = \Delta v\,\hat{\mathbf v},\qquad t_H = \pi\sqrt{a^3/\mu}$$

**Index (tensor) form**

$$\ddot x_{bi} = -\mu\,x_{bi}\,(x_{bj}x_{bj})^{-3/2},\qquad \tfrac12 v_{bi}v_{bi} - \frac{\mu}{\sqrt{x_{bj}x_{bj}}} = -\frac{\mu}{2a_b},\qquad v_{bi} \leftarrow v_{bi} + \Delta v_{bk}\,\frac{v_{bi}}{\sqrt{v_{bj}v_{bj}}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bi}$ | `y[:, 0, :]` | (B, 2) | position of spacecraft $b$ relative to the centre of the body, in km |
| $v_{bi}$ | `y[:, 1, :]` | (B, 2) | velocity of spacecraft $b$, in km/s |
| $x_{bj}x_{bj}$ | `einsum('bj,bj->b', x, x)` | (B,) | repeated $j$ ⇒ summed: squared distance $r_b^2$ |
| $\ddot x_{bi}$ | `acceleration(x, mu)` | (B, 2) | inverse-square pull toward the centre, $\mu = GM$ |
| $\sqrt{\mu(2/r - 1/a)}$ | `vis_viva(r, a, mu)` | like r | speed at radius $r$ on an orbit of semi-major axis $a$; every $\Delta v$ is a difference of two of these |
| $t_{bk},\; \Delta v_{bk}$ | `burn_schedule(...)` | (B, 3) | time and signed size of burn $k$ of spacecraft $b$ ($\Delta v < 0$ brakes) |
| $v_{bi} \leftarrow v_{bi} + \Delta v\,\hat v_{bi}$ | `apply_burn(y, dv)` | (B, 2, 2) | impulsive burn along the velocity, applied between two RK4 pieces |
| $E_b$ | `energy(y, mu)` | (T, B) | conserved between burns; each burn jumps it to the next orbit's $-\mu/2a$ |
| $\partial\,\Delta v_\mathrm{tot} / \partial r_b$ | `autograd.grad(dv1 + dv2 - dv3, rb)` | () | slope at $r_b = r_2$; it turns negative above $r_2/r_1 \approx 15.58$ |

The Hohmann transfer flies half of the ellipse that touches both circles, $a = (r_1 + r_2)/2$: a prograde burn $\Delta v_1$ at periapsis and a second one $\Delta v_2$ at apoapsis to circularise, after $t_H = \pi\sqrt{a^3/\mu}$. The bi-elliptic transfer first climbs to a far radius $r_b$, where the spacecraft is slow and a small burn raises the other side of the orbit to $r_2$, then brakes on arrival. It takes much longer, but for $r_2/r_1 \gtrsim 11.94$ a far enough $r_b$ makes it cheaper (the limit $r_b \to \infty$ costs $(\sqrt2 - 1)(v_{c1} + v_{c2})$), and above $r_2/r_1 \approx 15.58$ every $r_b > r_2$ does. Both crossovers are found numerically by bisection, the second with an autograd derivative. The default is low Earth orbit (about 320 km up) to geostationary orbit: about 3.9 km/s and 5.3 h by Hohmann. Choose the Sun with $r_1 = 1$ AU and $r_2 = 1.524$ AU for Earth → Mars: roughly 5.6 km/s heliocentric in total and about 259 days (the run always prints this case). Real missions also climb out of Earth's gravity well and must wait for the right planetary alignment; the orbits here are circular and coplanar and the burns are instantaneous. The run splits the RK4 integration at every burn so that each burn fires exactly at an apsis.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/orbit_transfer/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    body = tensors.BODIES[params["body"]]
    mu, unit, t_unit, t_label = body["mu"], body["unit_km"], body["t_unit"], body["t_label"]
    r1, r2, rb = tensors.radii_km(params, body)
    kinds = tensors.spacecraft_kinds(params["transfer"])

    # Fly every spacecraft at once; coast for half a target orbit after the last burn.
    y0 = tensors.initial_state(r1, len(kinds), mu, dev)  # (B, 2, 2)
    burn_t, burn_dv = tensors.burn_schedule(kinds, r1, r2, rb, mu, dev)  # (B, 3) each
    T_inner = 2 * torch.pi * (min(r1, r2) ** 3 / mu) ** 0.5  # period of the tighter circle
    t_end = float(burn_t.max()) + 0.5 * 2 * torch.pi * (r2**3 / mu) ** 0.5
    dt = max(T_inner / (60 if fast_mode() else 250), t_end / 60000)  # ≥ 250 steps per inner orbit, ≤ 60k steps
    times, traj, burn_pos = solve.fly(y0, mu, burn_t, burn_dv, t_end, dt)

    print(f"central body: {params['body']}   r1 = {r1:,.0f} km   r2 = {r2:,.0f} km   r2/r1 = {r2 / r1:.3f}")
    r = traj[:, :, 0, :].norm(dim=-1)  # (T, B)
    E = equation.energy(traj, mu)  # (T, B)
    for b, kind in enumerate(kinds):
        dv, tb = burn_dv[b], burn_t[b]
        k_arr = int(torch.searchsorted(times, tb[-1]))  # sample right at the last burn = arrival
        coast = times > tb[-1]
        between = (times > tb[0]) & (times < tb[1])  # first coasting arc (the transfer ellipse)
        drift = (E[between, b] / E[between, b][0] - 1).abs().max()
        spread = (r[coast, b].max() - r[coast, b].min()) / r2  # 0 for a perfectly circular final orbit
        burns = "  ".join(f"Δv{k + 1} = {float(v):+.3f}" for k, v in enumerate(dv) if v != 0) or "no burn needed"
        total, t_arr = float(dv.abs().sum()), float(tb[-1]) / t_unit
        print(f"{kind:>11}: {burns} km/s   total = {total:.3f} km/s   time = {t_arr:.2f} {t_label}")
        print(
            f"{'':>11}  integrated: r(arrival)/r2 = {float(r[k_arr, b]) / r2:.6f}"
            f"   final orbit (r_max - r_min)/r2 = {float(spread):.1e}   ΔE/E on the first ellipse = {float(drift):.1e}"
        )

    R_any, R_every = solve.crossover_any_rb(), solve.crossover_every_rb()
    print(f"bi-elliptic beats Hohmann above r2/r1 ≈ {R_any:.2f} (r_b → ∞) and for every r_b > r2 above ≈ {R_every:.2f}")

    sun = tensors.BODIES["sun"]
    au = torch.tensor(sun["unit_km"], dtype=tensors.DTYPE)
    dv1, dv2, t_em = equation.hohmann(au, tensors.MARS_AU * au, sun["mu"])
    print(
        f"Earth → Mars Hohmann (heliocentric, circular coplanar orbits): Δv1 ≈ {float(dv1):.2f} km/s,"
        f" Δv2 ≈ {float(dv2):.2f} km/s, total ≈ {float(dv1 + dv2):.2f} km/s, flight ≈ {float(t_em) / 86400:.0f} days"
    )

    curves = solve.total_dv_curves(tensors.ratio_grid(), params["rb_ratio"])
    return plot.animate(
        times / t_unit,
        traj / unit,
        burn_t / t_unit,
        burn_dv,
        burn_pos / unit,
        kinds,
        (r1 / unit, r2 / unit, rb / unit),
        params["body"],
        body["unit"],
        t_label,
        tensors.ratio_grid(),
        curves,
        (R_any, R_every),
        params["rb_ratio"],
        out_dir / "orbit_transfer.gif",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/orbit_transfer/equation.py
r"""Physics → tensor translation.

A spacecraft coasting around a central body of gravitational parameter μ = GM feels only its gravity:

    \ddot{\mathbf r} = -\frac{\mu}{r^3}\,\mathbf r

In index notation for a batch of spacecraft b and components i ∈ {x, y}:

    \ddot x_{bi} = -\mu \,x_{bi}\,(x_{bj} x_{bj})^{-3/2}

A repeated index (j above) is summed — that is einsum('bj,bj->b'). Between burns the energy per unit mass

    E_b = ½ v_{bi} v_{bi} - μ / \sqrt{x_{bj} x_{bj}} = -μ / (2 a_b)

is constant. Solving it for the speed gives the vis-viva equation v² = μ (2/r - 1/a): the speed anywhere on an orbit
of semi-major axis a. Every Δv below is a difference of two vis-viva speeds at the same radius.

An impulsive burn changes the velocity instantly along its own direction (prograde for Δv > 0, retrograde for Δv < 0):

    v_{bi} \leftarrow v_{bi} + Δv_b \, v_{bi} / \sqrt{v_{bj} v_{bj}}
"""

import math

import torch


def acceleration(x: torch.Tensor, mu: float) -> torch.Tensor:
    r"""\ddot x_{bi} = -μ x_{bi} (x_{bj} x_{bj})^{-3/2}.   x: (B, 2) → (B, 2)"""
    r2 = torch.einsum("bj,bj->b", x, x)  # (B,)  squared distance to the centre
    return -mu * torch.einsum("b,bi->bi", r2**-1.5, x)


def rhs(y: torch.Tensor, mu: float) -> torch.Tensor:
    """dy/dt for the state y of shape (B, 2, 2): y[:, 0] = position, y[:, 1] = velocity."""
    return torch.stack([y[:, 1], acceleration(y[:, 0], mu)], dim=1)


def energy(y: torch.Tensor, mu: float) -> torch.Tensor:
    r"""E_b = ½ v_{bi} v_{bi} - μ / r_b.   y: (..., 2, 2) → (...)"""
    x, v = y[..., 0, :], y[..., 1, :]
    return 0.5 * torch.einsum("...i,...i->...", v, v) - mu / torch.einsum("...i,...i->...", x, x).sqrt()


def apply_burn(y: torch.Tensor, dv: torch.Tensor) -> torch.Tensor:
    r"""v_{bi} ← v_{bi} + Δv_b v_{bi} / |v_b|: an instant burn along the velocity.   y: (B, 2, 2), dv: (B,)"""
    x, v = y[:, 0], y[:, 1]
    speed = torch.einsum("bj,bj->b", v, v).sqrt()  # (B,)
    v_new = v + torch.einsum("b,bi->bi", dv / speed, v)
    return torch.stack([x, v_new], dim=1)


# --- closed-form transfer costs (all broadcast over tensors of radii) ---------------------------------


def vis_viva(r: torch.Tensor, a: torch.Tensor, mu: float) -> torch.Tensor:
    r"""Speed at radius r on an orbit of semi-major axis a:  v = \sqrt{μ (2/r - 1/a)}."""
    return torch.sqrt(mu * (2 / r - 1 / a))


def circular_speed(r: torch.Tensor, mu: float) -> torch.Tensor:
    """Vis-viva with a = r."""
    return torch.sqrt(mu / r)


def half_period(a: torch.Tensor, mu: float) -> torch.Tensor:
    r"""Time to fly half an ellipse, periapsis → apoapsis:  t = π \sqrt{a^3 / μ}."""
    return math.pi * torch.sqrt(a**3 / mu)


def hohmann(r1: torch.Tensor, r2: torch.Tensor, mu: float):
    """Two burns via the ellipse that touches both circles. Returns signed (Δv₁, Δv₂) and the transfer time."""
    a = (r1 + r2) / 2  # transfer ellipse: periapsis r1, apoapsis r2 (or the other way round)
    dv1 = vis_viva(r1, a, mu) - circular_speed(r1, mu)  # leave the inner circle
    dv2 = circular_speed(r2, mu) - vis_viva(r2, a, mu)  # circularise on arrival
    return dv1, dv2, half_period(a, mu)


def bi_elliptic(r1: torch.Tensor, r2: torch.Tensor, rb: torch.Tensor, mu: float):
    """Three burns via two half-ellipses that meet far out at r_b. Returns signed (Δv₁, Δv₂, Δv₃) and the time."""
    a1 = (r1 + rb) / 2  # first ellipse: r1 → r_b
    a2 = (rb + r2) / 2  # second ellipse: r_b → r2
    dv1 = vis_viva(r1, a1, mu) - circular_speed(r1, mu)  # out to r_b
    dv2 = vis_viva(rb, a2, mu) - vis_viva(rb, a1, mu)  # at r_b: raise the far side to r2
    dv3 = circular_speed(r2, mu) - vis_viva(r2, a2, mu)  # circularise (a braking burn)
    return dv1, dv2, dv3, half_period(a1, mu) + half_period(a2, mu)


def bi_parabolic(r1: torch.Tensor, r2: torch.Tensor, mu: float) -> torch.Tensor:
    r"""Total Δv of the bi-elliptic transfer in the limit r_b → ∞:  (\sqrt2 - 1)(v_{c1} + v_{c2})."""
    return (math.sqrt(2) - 1) * (circular_speed(r1, mu) + circular_speed(r2, mu))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/orbit_transfer/tensors.py
"""Build the concrete tensors: central body, initial state, burn schedule and the grid of radius ratios.

Units: kilometres and seconds, so μ is in km³/s² and every Δv comes out in km/s. Radii are entered in the
body's own unit (Earth radii around the Earth, astronomical units around the Sun). Everything is float64:
the Sun's μ ≈ 1.3e11 km³/s² and radii ≈ 1e8 km would lose too many digits in float32.
"""

import torch

from . import equation

DTYPE = torch.float64

BODIES = {
    #        μ (km³/s²)           length unit (km)      unit label   time unit (s)
    "earth": {"mu": 398600.4418, "unit_km": 6378.137, "unit": "R⊕", "t_unit": 3600.0, "t_label": "h"},
    "sun": {"mu": 1.32712440018e11, "unit_km": 1.495978707e8, "unit": "AU", "t_unit": 86400.0, "t_label": "d"},
}
MARS_AU = 1.523679  # semi-major axis of Mars's orbit (Earth's is 1 AU by definition)


def radii_km(params: dict, body: dict) -> tuple[float, float, float]:
    """r1, r2 and the bi-elliptic intermediate radius r_b = (r_b/r2) · max(r1, r2), all in km."""
    r1 = params["r1"] * body["unit_km"]
    r2 = params["r2"] * body["unit_km"]
    rb = params["rb_ratio"] * max(r1, r2)
    return r1, r2, rb


def spacecraft_kinds(transfer: str) -> list[str]:
    """Which transfer each spacecraft in the batch flies."""
    return ["hohmann", "bi-elliptic"] if transfer == "both" else [transfer.replace("_", "-")]


def initial_state(r1: float, n: int, mu: float, device: torch.device) -> torch.Tensor:
    r"""y0 of shape (B, 2, 2): every spacecraft on the circle r1 at (r1, 0), moving in +y at v_c = \sqrt{μ / r1}."""
    zero = torch.zeros(n, dtype=DTYPE, device=device)
    position = torch.stack([zero + r1, zero], dim=-1)  # (B, 2)
    velocity = torch.stack([zero, zero + (mu / r1) ** 0.5], dim=-1)  # (B, 2)
    return torch.stack([position, velocity], dim=1)


def burn_schedule(kinds: list[str], r1: float, r2: float, rb: float, mu: float, device: torch.device):
    """Burn times t_bk and signed Δv_bk (km/s), both (B, 3), k = burn number.

    Every first burn is at t = 0 (on the +x axis). The Hohmann transfer pads its missing third burn with Δv = 0.
    """
    R1, R2, RB = (torch.tensor(r, dtype=DTYPE) for r in (r1, r2, rb))
    times, dvs = [], []
    for kind in kinds:
        if kind == "hohmann":
            dv1, dv2, t = equation.hohmann(R1, R2, mu)
            times.append([0.0, t, t])
            dvs.append([dv1, dv2, 0.0])
        else:
            dv1, dv2, dv3, t = equation.bi_elliptic(R1, R2, RB, mu)
            t1 = equation.half_period((R1 + RB) / 2, mu)
            times.append([0.0, t1, t])
            dvs.append([dv1, dv2, dv3])
    return _table(times, device), _table(dvs, device)


def _table(rows: list[list], device: torch.device) -> torch.Tensor:
    return torch.tensor([[float(v) for v in row] for row in rows], dtype=DTYPE, device=device)


def ratio_grid(n: int = 400) -> torch.Tensor:
    """Radius ratios R = r2/r1 from 1 to 100, log-spaced, (N,)."""
    return torch.logspace(0, 2, n, dtype=DTYPE)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/orbit_transfer/solve.py
"""Fly the whole batch with RK4, burning at the scheduled apsides, and compare transfer costs over radius ratios."""

import math

import torch

from problems._common.integrators import integrate

from . import equation


def fly(y0: torch.Tensor, mu: float, burn_t: torch.Tensor, burn_dv: torch.Tensor, t_end: float, dt: float):
    """Coast between burns with RK4 and apply each impulsive burn exactly on time.

    y0: (B, 2, 2), burn_t / burn_dv: (B, K). The run is split at every burn time of any spacecraft; each piece uses a
    step close to dt that ends exactly on the next burn. Returns times (T,), trajectory (T, B, 2, 2) and the burn
    positions (B, K, 2). At a burn time the trajectory stores the state just after the burn.
    """
    events = sorted({float(t) for t in burn_t.flatten()} | {t_end})
    burn_pos = torch.zeros(*burn_t.shape, 2, dtype=y0.dtype, device=y0.device)
    times, pieces, y = [], [], y0
    for t0, t1 in zip(events[:-1], events[1:]):
        now = (burn_t - t0).abs() < 1e-9 * max(1.0, t_end)  # (B, K)  which burns happen at t0
        burn_pos[now] = y[:, 0].unsqueeze(1).expand_as(burn_pos)[now]
        y = equation.apply_burn(y, (burn_dv * now).sum(dim=1))  # the padded Hohmann burn adds Δv = 0
        n = max(1, math.ceil((t1 - t0) / dt))
        piece = integrate(lambda s, t: equation.rhs(s, mu), y, (t1 - t0) / n, n)  # (n + 1, B, 2, 2)
        pieces.append(piece[:-1])
        times.append(t0 + torch.arange(n, dtype=y0.dtype, device=y0.device) * (t1 - t0) / n)
        y = piece[-1]
    pieces.append(y[None])
    times.append(torch.tensor([t_end], dtype=y0.dtype, device=y0.device))
    return torch.cat(times), torch.cat(pieces), burn_pos


def total_dv_curves(R: torch.Tensor, rb_ratio: float) -> dict[str, torch.Tensor]:
    """Total Δv / v_c1 for r1 = 1, μ = 1 over the radius ratios R (N,): Hohmann, bi-elliptic with r_b = rb_ratio · r2,
    and the bi-parabolic limit r_b → ∞."""
    one = torch.ones_like(R)
    dv1, dv2, _ = equation.hohmann(one, R, 1.0)
    b1, b2, b3, _ = equation.bi_elliptic(one, R, rb_ratio * R, 1.0)
    return {
        "hohmann": dv1.abs() + dv2.abs(),
        "bi-elliptic": b1.abs() + b2.abs() + b3.abs(),
        "bi-parabolic": equation.bi_parabolic(one, R, 1.0),
    }


def bisect(f, lo: float, hi: float, n_iter: int = 60) -> float:
    """Root of a scalar function that changes sign between lo and hi."""
    f_lo = f(lo)
    for _ in range(n_iter):
        mid = 0.5 * (lo + hi)
        if (f(mid) > 0) == (f_lo > 0):
            lo, f_lo = mid, f(mid)
        else:
            hi = mid
    return 0.5 * (lo + hi)


def crossover_any_rb() -> float:
    """r2/r1 above which a far enough r_b makes the bi-elliptic transfer cheaper: Hohmann = bi-parabolic (≈ 11.94)."""

    def gap(R: float) -> float:
        one, r2 = torch.tensor(1.0, dtype=torch.float64), torch.tensor(R, dtype=torch.float64)
        dv1, dv2, _ = equation.hohmann(one, r2, 1.0)
        return float(dv1 + dv2 - equation.bi_parabolic(one, r2, 1.0))

    return bisect(gap, 2.0, 50.0)


def crossover_every_rb() -> float:
    """r2/r1 above which every r_b > r2 beats Hohmann (≈ 15.58).

    At r_b = r2 the bi-elliptic transfer is the Hohmann transfer; it becomes cheaper for every r_b once raising r_b
    a little lowers the cost, i.e. once d(Δv_total)/d r_b < 0 at r_b = r2. Autograd gives that derivative.
    """

    def slope(R: float) -> float:
        r2 = torch.tensor(R, dtype=torch.float64)
        rb = r2.clone().requires_grad_(True)
        dv1, dv2, dv3, _ = equation.bi_elliptic(torch.tensor(1.0, dtype=torch.float64), r2, rb, 1.0)
        (grad,) = torch.autograd.grad(dv1 + dv2 - dv3, rb)  # Δv₃ < 0 is a braking burn, so its size is -Δv₃
        return float(grad)

    return bisect(slope, 2.0, 50.0)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/orbit_transfer/plot.py
"""Animate the transfers (orbits, transfer ellipses, spacecraft, burns) next to total Δv against r2/r1."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Circle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

from problems._common.plotting import ACCENT, frame_indices, save_gif

GREY = "#9ca3af"


def ellipse(x_plus: float, x_minus: float, n: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Orbit with the central body at a focus, crossing the +x axis at x_plus and the -x axis at -x_minus."""
    theta = np.linspace(0, 2 * np.pi, n)
    a, centre, b = (x_plus + x_minus) / 2, (x_plus - x_minus) / 2, np.sqrt(x_plus * x_minus)
    return centre + a * np.cos(theta), b * np.sin(theta)


def animate(
    times: torch.Tensor,
    traj: torch.Tensor,
    burn_t: torch.Tensor,
    burn_dv: torch.Tensor,
    burn_pos: torch.Tensor,
    kinds: list[str],
    radii: tuple[float, float, float],
    body: str,
    unit: str,
    t_label: str,
    R: torch.Tensor,
    curves: dict[str, torch.Tensor],
    crossovers: tuple[float, float],
    rb_ratio: float,
    path: Path,
) -> Path:
    t = times.cpu().numpy()  # (T,)
    pos = traj[:, :, 0, :].cpu().numpy()  # (T, B, 2)
    bt, bdv, bpos = burn_t.cpu().numpy(), burn_dv.cpu().numpy(), burn_pos.cpu().numpy()  # (B, 3), (B, 3), (B, 3, 2)
    r1, r2, rb = radii
    B = pos.shape[1]
    colours = [ACCENT[0] if k == "hohmann" else ACCENT[2] for k in kinds]

    fig, (ax, ax_dv) = plt.subplots(1, 2, figsize=(8.6, 4), gridspec_kw={"width_ratios": [1.25, 1]})
    ax.set_aspect("equal", adjustable="datalim")
    unit = r"$R_\oplus$" if unit == "R⊕" else unit  # mathtext: the ⊕ glyph is missing from the default font
    ax.set_xlabel(f"x ({unit})")
    ax.set_ylabel(f"y ({unit})")

    # start and target circles, and the transfer ellipses (dashed)
    for r, label in [(r1, "$r_1$"), (r2, "$r_2$")]:
        ax.plot(*ellipse(r, r), color="#374151", lw=1.2)
        ax.annotate(label, (r * 0.71, -r * 0.71), xytext=(4, -4), textcoords="offset points", fontsize=10, va="top")
    for k, kind in enumerate(kinds):
        legs = [(r1, r2)] if kind == "hohmann" else [(r1, rb), (r2, rb)]
        for x_plus, x_minus in legs:
            ax.plot(*ellipse(x_plus, x_minus), "--", color=colours[k], lw=0.9, alpha=0.6)
    if body == "earth":
        ax.add_patch(Circle((0, 0), 1.0, color="#93c5fd", ec="#1e40af", lw=1, zorder=4))  # Earth to scale
    else:
        ax.plot([0], [0], "*", ms=14, color="#f59e0b", zorder=4)  # the Sun (much smaller than these orbits)

    trails = [ax.plot([], [], lw=1.6, color=colours[b])[0] for b in range(B)]
    dots = [
        ax.plot([], [], "o", ms=6, color=colours[b], label=f"{kinds[b]}: Σ|Δv| = {np.abs(bdv[b]).sum():.2f} km/s")[0]
        for b in range(B)
    ]
    burns = []  # (time, marker) pairs: a red cross appears where and when a burn fires
    for b in range(B):
        for k in range(bt.shape[1]):
            if bdv[b, k] != 0:
                (mark,) = ax.plot([bpos[b, k, 0]], [bpos[b, k, 1]], "X", ms=8, color="#dc2626", mec="white", zorder=6)
                mark.set_visible(False)
                burns.append((bt[b, k], mark))
    ax.legend(loc="lower left", fontsize=7, framealpha=0.9)
    clock = ax.text(0.03, 0.97, "", transform=ax.transAxes, va="top", fontsize=9)
    ax.set_title("× = impulsive burn (prograde or retrograde)", fontsize=9)

    # total cost against the radius ratio, in units of the starting circular speed
    Rn = R.cpu().numpy()
    ax_dv.semilogx(Rn, curves["hohmann"].cpu().numpy(), color=ACCENT[0], lw=1.8, label="Hohmann")
    ax_dv.semilogx(
        Rn,
        curves["bi-elliptic"].cpu().numpy(),
        color=ACCENT[2],
        lw=1.5,
        label=f"bi-elliptic, $r_b = {rb_ratio:g}\\,r_2$",
    )
    ax_dv.semilogx(
        Rn, curves["bi-parabolic"].cpu().numpy(), ":", color=ACCENT[3], lw=1.5, label=r"bi-elliptic, $r_b\to\infty$"
    )
    for x, text in zip(crossovers, ["11.94", "15.58"]):
        ax_dv.axvline(x, color=GREY, ls="--", lw=1)
        ax_dv.text(
            x * 1.04,
            0.98,
            text,
            fontsize=8,
            color="#4b5563",
            rotation=90,
            va="top",
            transform=ax_dv.get_xaxis_transform(),
        )
    ratio = r2 / r1
    if 1 <= ratio <= 100:
        k = int(np.argmin(np.abs(Rn - ratio)))
        for name in ["hohmann", "bi-elliptic"]:
            ax_dv.plot([ratio], [float(curves[name][k])], "o", ms=6, mfc="white", mec="#111827", zorder=5)
    ax_dv.set_xlim(1, 100)
    ax_dv.xaxis.set_major_locator(
        FixedLocator([1, 2, 5, 10, 20, 50, 100])
    )  # plain labels, no minor ticks: faster frames
    ax_dv.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:g}"))
    ax_dv.xaxis.set_minor_locator(NullLocator())
    top = max(float(c.max()) for c in curves.values())
    ax_dv.set_ylim(0, min(1.12 * top, 0.9))
    ax_dv.set_xlabel("$r_2 / r_1$")
    ax_dv.set_ylabel(r"total $\Delta v$ / $v_{c1}$")
    ax_dv.legend(fontsize=7, loc="lower right")
    ax_dv.set_title("cost of the transfer (○ = this run)", fontsize=9)

    def update(step: int):
        for b in range(B):
            trails[b].set_data(pos[: step + 1, b, 0], pos[: step + 1, b, 1])
            dots[b].set_data([pos[step, b, 0]], [pos[step, b, 1]])
        for when, mark in burns:
            mark.set_visible(t[step] >= when)
        clock.set_text(f"t = {t[step]:.1f} {t_label}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'body': 'earth',
    'r1': 1.05,
    'r2': 6.61,
    'transfer': 'both',
    'rb_ratio': 2.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.orbit_transfer")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.orbit_transfer.main")
problem_dir = Path("problems/astro/orbit_transfer")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
